In [5]:
import cv2
import os
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle

SCOPES = ['https://www.googleapis.com/auth/drive.file']

# --- authenticate ---
creds = None
if os.path.exists('token.pickle'):
    with open('token.pickle', 'rb') as token:
        creds = pickle.load(token)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            'credentials.json', SCOPES)
        creds = flow.run_local_server(port=0)

    with open('token.pickle', 'wb') as token:
        pickle.dump(creds, token)

service = build('drive', 'v3', credentials=creds)

# --- capture frame ---
cap = cv2.VideoCapture(1200)
ret, frame = cap.read()
cap.release()

filename = "frame.jpg"
cv2.imwrite(filename, frame)

# --- upload ---
file_metadata = {"name": filename}
media = MediaFileUpload(filename, mimetype="image/jpeg")

file = service.files().create(
    body=file_metadata,
    media_body=media,
    fields="id"
).execute()

print("Uploaded file ID:", file.get("id"))

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=40054076548-r602epe3h1qvo1q8bh4b6sjmoppvov62.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A53553%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=sYu9iPoZLlSz63eRpgdVpNzCWuIXEk&code_challenge=jTFip6KwUYChNDw5ySG4gT3iNDsdrWmpUBBVuVdTAGI&code_challenge_method=S256&access_type=offline
Uploaded file ID: 1GAngPbT4IDC6ZCnYjUCG4eUzTO_82lGc
